# Transcription Adapter

> The typed transcription task contract — `TranscriptionAdapter` ABC +
> the provisional tool protocol (capability-unit Option C, pass-2 Thread 3).

In [ ]:
#| default_exp adapter

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from abc import abstractmethod
from pathlib import Path
from typing import Any, ClassVar, Protocol, Union, runtime_checkable

from cjm_plugin_system.core.adapter import TaskAdapter

from cjm_transcription_adapter_interface.core import TranscriptionResult

## Tool protocol (provisional)

`required_tool_protocol` is declared with a PROVISIONAL protocol (the Q5
posture: declare the slot, let tool-splitting evidence finalize the body).
It mirrors the fused-era tool surface — the task-shaped `execute` every
current transcription capability exposes — so stage-4 surface-based
compatibility matching has a real contract to match against recorded
structural surfaces. When the stage-8 Option C cascade splits tools, the
protocol is RE-DERIVED from the native tool surfaces (whisper
`transcribe`, voxtral `apply_transcription_request`, ...).

In [ ]:
#| export
@runtime_checkable
class TranscriptionToolProtocol(Protocol):
    """PROVISIONAL structural contract for transcription-capable tools.

    Mirrors the fused-era surface (task-shaped `execute`); re-derived from
    native tool surfaces when the Option C cascade splits tools (stage 8).
    """
    def execute(self, audio: Union[str, Path], **kwargs) -> Any: ...


In [ ]:
#| export
class TranscriptionAdapter(TaskAdapter):
    """Typed transcription task adapter: model-ready audio in,
    `TranscriptionResult` out.

    Input contract (carried over from the fused-era TranscriptionPlugin):
    the caller guarantees MODEL-READY audio — format / sample-rate /
    channel handling happens upstream (ffmpeg `convert_for_model`), never
    in the adapter.

    Persistence sits BESIDE the task method (pass-2 Thread 3): the storage
    module's `TranscriptionStorage` provides the adapter-level cache /
    persist seam (`get_cached(audio_path, audio_hash, config_hash)` +
    `save_with_logging(...)`).

    The result DTO is wire-registered ("transcription.result"): returned
    values cross the worker boundary typed via the substrate's `core.wire`
    envelope.
    """

    task_name: ClassVar[str] = "transcription"
    required_tool_protocol: ClassVar[type] = TranscriptionToolProtocol  # PROVISIONAL

    @abstractmethod
    def transcribe(
        self,
        audio: Union[str, Path],  # Path to MODEL-READY audio (converted upstream)
        **kwargs,                 # Adapter-specific options (language, task, ...)
    ) -> TranscriptionResult:     # Typed transcription output
        """Transcribe model-ready audio to text."""
        ...


In [ ]:
# Shape tests: abstract gate + ClassVars + a concrete impl returning the
# typed DTO + the provisional protocol structurally matching a fused-era
# tool shape.
class _FakeTranscriptionAdapter(TranscriptionAdapter):
    def transcribe(self, audio, **kwargs) -> TranscriptionResult:
        return TranscriptionResult(text=f"transcribed:{audio}")


try:
    TranscriptionAdapter()  # type: ignore[abstract]
    raise AssertionError("abstract ABC must not instantiate")
except TypeError:
    pass

impl = _FakeTranscriptionAdapter()
out = impl.transcribe("a.wav")
assert isinstance(out, TranscriptionResult) and out.text == "transcribed:a.wav"
assert TranscriptionAdapter.task_name == "transcription"
assert TranscriptionAdapter.required_tool_protocol is TranscriptionToolProtocol


class _FusedEraTool:  # structural match, no inheritance
    def execute(self, audio, **kwargs):
        return {"text": "hi"}


assert isinstance(_FusedEraTool(), TranscriptionToolProtocol)
print("TranscriptionAdapter shape tests OK")

TranscriptionAdapter shape tests OK


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()